The RAG process is fixed: every request is used as the basis for a database search for matching documents. The information found in the matching documents is then used to provide a detailed output.
With agents, the LLM agent is given the *option* to perform a database search for answering a given prompt.

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
import json

In [6]:
load_dotenv()
openai_client = OpenAI()

In [7]:
documents = load_faq_data()
index = build_index(documents)

In [8]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
  )

In [12]:
search_tool = {
	"type": "function",
	'function': {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [13]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [14]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	tools=[search_tool],
	user="llm-zoomcamp",
	stream=False
)

In [18]:
if response.choices[0].finish_reason == "tool_calls":
	function_call = response.choices[0].message.tool_calls[0].function

	if function_call.name == "search":
		search_keyword = json.loads(function_call.arguments)['query']
		print(search_keyword)
		search_results = search(search_keyword)


join course
